# md_combat — Python vs R comparison

This notebook demonstrates all three Python implementations side-by-side with their R equivalents
and confirms that results are numerically consistent.

**Sections**
1. Setup
2. ComBat (microarray) — Python vs R
3. ComBatSeq (RNA-seq) — Python vs R
4. ComBatSeqFast (RNA-seq, vectorised NR) — Python vs R
5. Fast vs Standard — direct comparison
6. Summary table

**Requirements**: R with `sva`, `bladderbatch`, `Biobase`, `airway`, `SummarizedExperiment` installed.
See `CLAUDE.md` for R setup instructions.  Python packages are installed via `uv sync --extra dev`.

## 1. Setup

In [ ]:
import shutil
import subprocess
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

from md_combat.combat import ComBat
from md_combat.combat_seq import ComBatSeq, ComBatSeqFast
from md_combat.datasets import load_airway, load_bladderbatch

EXPECTED_DIR = Path("../tests/expected")
_RSCRIPT_OPTS = ["--no-save", "--no-restore", "--quiet"]


def _find_rscript():
    for name in ["Rscript", "Rscript4.5", "Rscript4.4", "Rscript4.3"]:
        path = shutil.which(name)
        if path:
            return path
    return None


def run_r(script: str, timeout: int = 300) -> str:
    """Run an R script string and return stdout; raise on failure."""
    rscript = _find_rscript()
    if rscript is None:
        raise RuntimeError("Rscript not found on PATH")
    result = subprocess.run(
        [rscript, *_RSCRIPT_OPTS, "-e", script],
        capture_output=True, text=True, timeout=timeout,
    )
    if result.returncode != 0:
        raise RuntimeError(f"R failed:\n{result.stderr}")
    return result.stdout


def pearson_r(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.corrcoef(a.ravel().astype(float), b.ravel().astype(float))[0, 1])


RSCRIPT = _find_rscript()
print(f"Rscript: {RSCRIPT}")
print(f"R version: {run_r('cat(R.version$version.string)')}")

## 2. ComBat — Python vs R

**Dataset**: bladderbatch microarray (22 283 genes × 57 samples, 5 batches)  
**Expected**: near-exact numerical parity (rtol ≤ 1e-5)

In [ ]:
# --- load data ---
expr_df, batch_bb, _ = load_bladderbatch()
print(f"bladderbatch: {expr_df.shape[0]} genes × {expr_df.shape[1]} samples, {len(set(batch_bb))} batches")

In [ ]:
# --- Python ---
py_combat = ComBat().fit_transform(expr_df, batch_bb)
print(f"Python ComBat: {py_combat.shape}")

In [ ]:
# --- R (uses pre-computed golden file) ---
r_combat = pd.read_parquet(EXPECTED_DIR / "bladderbatch_combat.parquet")
print(f"R sva::ComBat: {r_combat.shape}")

In [ ]:
# --- parity check ---
max_diff = np.abs(py_combat.values - r_combat.values).max()
r = pearson_r(py_combat.values, r_combat.values)
print(f"Max absolute difference : {max_diff:.2e}")
print(f"Pearson r               : {r:.8f}")
print(f"\nVerdict: {'PASS ✓' if max_diff < 1e-3 else 'FAIL ✗'}  (expected max_diff < 1e-3)")

## 3. ComBatSeq — Python vs R

**Dataset**: airway RNA-seq 20K gene subset (20 000 genes × 8 samples, 4 cell-line batches)  
**Expected**: Pearson r > 0.95 (R uses edgeR C++ solver; Python uses statsmodels NM)

In [ ]:
# --- load 20K subset ---
counts_full, batch_aw, group_aw = load_airway()
gene_idx = pd.read_parquet(EXPECTED_DIR / "airway_20k_gene_ids.parquet")["gene_idx"].tolist()
counts_20k = counts_full.iloc[gene_idx]
print(f"airway 20K subset: {counts_20k.shape[0]} genes × {counts_20k.shape[1]} samples")

In [ ]:
# --- Python ComBatSeq (standard NM solver) ---
# Note: this takes ~3 min due to per-gene statsmodels optimisation
py_seq_std = ComBatSeq().fit_transform(counts_20k, batch_aw, group=group_aw)
print(f"Python ComBatSeq: {py_seq_std.shape}")

In [ ]:
# --- R golden file ---
r_seq = pd.read_parquet(EXPECTED_DIR / "airway_20k_combat_seq.parquet")
print(f"R sva::ComBat_seq: {r_seq.shape}")

In [ ]:
# --- parity check ---
r_std = pearson_r(py_seq_std.values, r_seq.values)
print(f"Pearson r (Python ComBatSeq vs R): {r_std:.4f}")
print(f"\nVerdict: {'PASS ✓' if r_std > 0.95 else 'FAIL ✗'}  (expected r > 0.95)")

## 4. ComBatSeqFast — Python vs R

**Same 20K airway subset.**  
**Expected**: Pearson r > 0.95.  Fast is the production-recommended path for large datasets.

In [ ]:
# --- Python ComBatSeqFast (vectorised Newton-Raphson) ---
py_seq_fast = ComBatSeqFast().fit_transform(counts_20k, batch_aw, group=group_aw)
print(f"Python ComBatSeqFast: {py_seq_fast.shape}")

In [ ]:
# --- parity check vs R ---
r_fast = pearson_r(py_seq_fast.values, r_seq.values)
print(f"Pearson r (ComBatSeqFast vs R): {r_fast:.4f}")
print(f"\nVerdict: {'PASS ✓' if r_fast > 0.95 else 'FAIL ✗'}  (expected r > 0.95)")

## 5. Fast vs Standard — direct comparison

Both Python implementations should be highly correlated (r > 0.99).  
The standard NM solver has convergence failures on real data; `ComBatSeqFast` is more reliable.

In [ ]:
r_fvs = pearson_r(py_seq_fast.values, py_seq_std.values)
exact_match = (py_seq_fast.values == py_seq_std.values).mean() * 100

print(f"Pearson r (Fast vs Standard)     : {r_fvs:.4f}")
print(f"Exact integer match              : {exact_match:.1f}%")
print(f"\nVerdict: {'PASS ✓' if r_fvs > 0.99 else 'FAIL ✗'}  (expected r > 0.99)")

## 6. Summary table

In [ ]:
summary = pd.DataFrame([
    {
        "comparison": "ComBat Python vs R",
        "dataset": "bladderbatch (22K genes)",
        "metric": "max |diff|",
        "value": f"{max_diff:.2e}",
        "threshold": "< 1e-3",
        "pass": max_diff < 1e-3,
    },
    {
        "comparison": "ComBatSeq Python vs R",
        "dataset": "airway 20K genes",
        "metric": "Pearson r",
        "value": f"{r_std:.4f}",
        "threshold": "> 0.95",
        "pass": r_std > 0.95,
    },
    {
        "comparison": "ComBatSeqFast Python vs R",
        "dataset": "airway 20K genes",
        "metric": "Pearson r",
        "value": f"{r_fast:.4f}",
        "threshold": "> 0.95",
        "pass": r_fast > 0.95,
    },
    {
        "comparison": "ComBatSeqFast vs ComBatSeq",
        "dataset": "airway 20K genes",
        "metric": "Pearson r",
        "value": f"{r_fvs:.4f}",
        "threshold": "> 0.99",
        "pass": r_fvs > 0.99,
    },
])

summary["status"] = summary["pass"].map({True: "PASS", False: "FAIL"})
display(summary.drop(columns="pass"))

overall = summary["pass"].all()
print(f"\nOverall: {'ALL PASS ✓' if overall else 'SOME FAILURES ✗'}")